# 🤖 IA local para UML Board (Ollama + Qwen2.5-VL en Google Colab)

Este cuaderno monta un **modelo de IA abierto** en una GPU gratuita de Colab y lo expone con un túnel
HTTPS para que la pizarra (`Motor de IA → Local`) lo use. No usa claves ni servicios de pago.

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4** y guarda.

**Cómo usarlo:** ejecuta las celdas en orden (o *Entorno de ejecución → Ejecutar todas*).
Al final aparece una URL `https://….trycloudflare.com`: cópiala en la pizarra
(Chat IA → ⚙️ Motor de IA → Local) y pulsa **Probar conexión**.

⚠️ La URL **cambia cada vez** que ejecutas el cuaderno, y la sesión de Colab se cierra sola tras un rato de
inactividad. Déjalo corriendo mientras lo uses.

## 1. Comprobar que hay GPU

In [ ]:
import subprocess

salida = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
print(salida.stdout or salida.stderr)
if "GPU" not in salida.stdout:
    raise SystemExit("⚠️ No hay GPU. Menú: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU, y vuelve a empezar.")
print("✅ GPU lista")

## 2. Instalar Ollama (el servidor de modelos)

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Iniciar el servidor

`OLLAMA_ORIGINS="*"` permite que la página web (en Vercel o `localhost`) le hable desde el navegador (CORS).
`OLLAMA_KEEP_ALIVE=-1` deja el modelo cargado en la GPU para que las respuestas no tarden en arrancar.

In [ ]:
import os, subprocess, time, urllib.request

# Modelo con visión (sirve para importar diagramas desde foto). Si la GPU no alcanza, usa "qwen2.5vl:3b".
MODELO = "qwen2.5vl:7b"

entorno = dict(
    os.environ,
    OLLAMA_HOST="0.0.0.0:11434",
    OLLAMA_ORIGINS="*",
    OLLAMA_KEEP_ALIVE="-1",
    OLLAMA_NUM_PARALLEL="1",
    OLLAMA_FLASH_ATTENTION="1",
)
servidor = subprocess.Popen(
    ["ollama", "serve"], env=entorno,
    stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT,
)

for _ in range(40):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("✅ Ollama activo")
        break
    except Exception:
        time.sleep(1)
else:
    raise SystemExit("Ollama no arrancó. Revisa /content/ollama.log")

## 4. Descargar el modelo (la primera vez tarda unos minutos, ~6 GB)

In [ ]:
!ollama pull $MODELO

## 5. Precalentar: cargar el modelo en la GPU (así la primera pregunta no espera)

In [ ]:
import json, urllib.request

pedido = urllib.request.Request(
    "http://127.0.0.1:11434/api/generate",
    data=json.dumps({"model": MODELO, "prompt": "Responde solo: listo", "stream": False, "keep_alive": -1}).encode(),
    headers={"Content-Type": "application/json"},
)
respuesta = json.load(urllib.request.urlopen(pedido, timeout=300))
print("Modelo respondió:", respuesta.get("response", "").strip())

## 6. Abrir el túnel HTTPS (Cloudflare, sin cuenta)

Convierte el servidor de Colab en una URL pública `https://….trycloudflare.com`.
`--http-host-header` evita que Ollama rechace el nombre de host del túnel.

In [ ]:
import re, subprocess, time

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

tunel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:11434",
     "--http-host-header", "127.0.0.1:11434"],
    stdout=open("/content/cloudflared.log", "w"), stderr=subprocess.STDOUT,
)

URL_PUBLICA = None
for _ in range(60):
    time.sleep(1)
    registro = open("/content/cloudflared.log").read()
    encontrada = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", registro)
    if encontrada:
        URL_PUBLICA = encontrada.group(0)
        break

if not URL_PUBLICA:
    raise SystemExit("No se obtuvo la URL del túnel. Revisa /content/cloudflared.log")

print("\n" + "=" * 60)
print("  URL PARA LA PIZARRA (Motor de IA → Local):")
print("  " + URL_PUBLICA)
print("  Modelo: " + MODELO)
print("=" * 60)

## 7. Comprobar el túnel desde fuera (y que el navegador podrá conectarse)

In [ ]:
import json, urllib.request

pedido = urllib.request.Request(URL_PUBLICA + "/api/tags", headers={"Origin": "https://ejemplo.vercel.app"})
with urllib.request.urlopen(pedido, timeout=30) as r:
    modelos = [m["name"] for m in json.load(r).get("models", [])]
    cors = r.headers.get("Access-Control-Allow-Origin")

print("Modelos disponibles:", modelos)
print("CORS (Access-Control-Allow-Origin):", cors)
print("✅ Túnel OK" if MODELO in modelos and cors else "⚠️ Revisa los pasos anteriores")

## 8. Mantener la sesión viva

Deja **esta celda ejecutándose** mientras uses la pizarra. Si se detiene (o Colab se desconecta),
vuelve a ejecutar desde el paso 3: la URL será otra.

In [ ]:
import time, urllib.request

print("Sesión activa. Detén esta celda para apagar la IA local.")
while True:
    time.sleep(240)
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5)
        print(time.strftime("%H:%M:%S"), "· IA local activa ·", URL_PUBLICA)
    except Exception as e:
        print(time.strftime("%H:%M:%S"), "· ⚠️ el servidor no responde:", e)